# Gender Wage Gap in the US, 1976-2025
CPS ASEC from IPUMS | Course research project

**Question.** How much has the gender wage gap closed in the US since 1976, and for whom?

**Rule:** every table and chart uses the survey weight, `asecwt`. Unweighted counts are only for checking data.

**Table first, chart second.** Print the weighted table (1 decimal), then chart exactly that table. Bar and line charts start at 0.

---
## 0. Setup
The file `jobs_assignment_data_20261104.csv` is in the root of My Drive: income years 1976-2025, ages 16-64, wage and salary workers with positive wages. The file has 85 columns and is 1.6 GB, so we load only the 18 below. `year` is the **income year** (the year the wages were earned). The setup cell adds a text label for each coded variable (`female_label`, `fulltime_label`, `educ5_label`, and so on).

| Variable | Meaning |
|---|---|
| `year`, `age`, `asecwt` | income year; age 16-64; weight (Americans each row stands for) |
| `female` | 1 = female, 0 = male |
| `fulltime`, `hrswk`, `wkswork` | full-time (1) or part-time (0) last year, full-time = 35+ usual weekly hours; usual weekly hours; weeks worked last year |
| `incwage_real`, `hrwage_real` | yearly wage and salary income; hourly wage (yearly wage / (weeks x hours)); both in 2025 dollars |
| `agegrp` | 16-24, 25-34, 35-44, 45-54, 55-64 |
| `educ5` | None/preschool, Less than HS, High school, Some college, Bachelor's+ |
| `raceeth` | White, Black (non-Hispanic), Hispanic (any race), Other; missing = don't know or no response |
| `marst3` | Married, Previously married, Never married |
| `nchild_cat`, `child_u5` | number of children (0, 1, 2, 3+); 1 = child under 5 |
| `occ6` | 6 occupation groups; missing = not in universe, military, 999, or other |
| `classwkr3` | Private, Government (the self-employed were removed) |
| `metro` | 1 = in a metro area, 0 = not; missing in some years |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount("/content/drive")

data_path = "/content/drive/My Drive/jobs_assignment_data_20261104.csv"        # change here if the file moves
cols = ["year", "age", "asecwt", "female", "fulltime", "hrswk", "wkswork", "incwage_real", "hrwage_real",
        "agegrp", "educ5", "raceeth", "marst3", "nchild_cat", "child_u5", "occ6", "classwkr3", "metro"]

df = pd.read_csv(data_path, usecols=cols)                                      # read only these columns: the file is big
print(df.shape)                                                                # rows, columns we loaded
print(list(df.columns))                                                        # their names
print(len(pd.read_csv(data_path, nrows=0).columns), "columns in the file")     # nrows=0 reads only the header

In [ ]:
# Labels for the codes, then a check: every code should land in exactly one label
df["female_label"] = df["female"].map({0: "Male", 1: "Female"})
df["fulltime_label"] = df["fulltime"].map({0: "Part-time", 1: "Full-time"})
df["agegrp_label"] = df["agegrp"].map({1: "16-24", 2: "25-34", 3: "35-44", 4: "45-54", 5: "55-64"})
df["educ5_label"] = df["educ5"].map({1: "None/preschool", 2: "Less than HS", 3: "High school", 4: "Some college", 5: "Bachelor's+"})
df["raceeth_label"] = df["raceeth"].map({1: "Hispanic", 2: "White", 3: "Black", 4: "Other"})        # missing stays missing
df["marst3_label"] = df["marst3"].map({1: "Married", 2: "Previously married", 3: "Never married"})
df["nchild_cat_label"] = df["nchild_cat"].map({0: "0", 1: "1", 2: "2", 3: "3+"})
df["child_u5_label"] = df["child_u5"].map({0: "No child under 5", 1: "Child under 5"})
df["occ6_label"] = df["occ6"].map({1: "Managers and professionals", 2: "Technical, sales and admin", 3: "Service",
                                   4: "Farming, forestry and fishing", 5: "Craft and repair", 6: "Operators and laborers"})   # missing stays missing
df["classwkr3_label"] = df["classwkr3"].map({1: "Private", 2: "Government"})
df["metro_label"] = df["metro"].map({0: "Non-metro", 1: "Metro"})                                  # missing stays missing
for name in ["female", "fulltime", "agegrp", "educ5", "raceeth", "marst3", "nchild_cat", "child_u5", "occ6", "classwkr3", "metro"]:
    print(pd.crosstab(df[name], df[name + "_label"], dropna=False))                                # each row should have one non-zero cell

In [ ]:
# Weighted mean: multiply each value by its weight, add up, divide by the sum of the weights
def wmean(data, var):                                                   # a table and a column name
    ok = data[var].notna()                                              # rows that have this column
    return np.average(data.loc[ok, var], weights=data.loc[ok, "asecwt"])   # the weighted mean

---
## 1. Summary table
**Question.** What does each loaded column look like before any math? The mean is weighted with `asecwt`. The count (n), the missing count, the min and the max are unweighted: they are data checks. A mean of a 0/1 column like `female` is a share. A mean of a code with several groups like `educ5` means nothing by itself; we keep the row to check n, missing, min and max.

In [ ]:
summary = pd.DataFrame({
    "n": df[cols].count(),                                  # people with a value
    "missing": df[cols].isna().sum(),                       # people without a value
    "weighted mean": [wmean(df, c) for c in cols],          # mean weighted with asecwt
    "min": df[cols].min(),                                  # smallest value
    "max": df[cols].max(),                                  # largest value
}, index=cols)
print(summary.round(1))

---
## 2. A new tool: weighted percentile
A percentile is the wage that a given share of workers earn less than. With weights, each row stands for `asecwt` people. Sort the rows by wage, add up the weights, and take the first wage where the running total reaches that share of all people. Four made-up people, so you can check by hand: the weights add up to 600, so the running shares are 17%, 33%, 83% and 100%.

In [ ]:
# Weighted percentile: sort by value, add up the weights, take the first value where the running share reaches q
def wquantile(data, var, q):                               # a table, a column name, and q between 0 and 1
    ok = data[var].notna()                                 # rows that have this column
    values = data.loc[ok, var].to_numpy()                  # the values
    weights = data.loc[ok, "asecwt"].to_numpy()            # their weights
    order = np.argsort(values)                             # positions that sort the values
    running = np.cumsum(weights[order]) / weights.sum()    # share of people at or below each value
    return values[order][np.searchsorted(running, q)]      # first value where the running share reaches q

example_df = pd.DataFrame({"wage": [10, 20, 30, 40], "asecwt": [100, 100, 300, 100]})   # 600 people
print(wquantile(example_df, "wage", 0.01), wquantile(example_df, "wage", 0.5), wquantile(example_df, "wage", 0.99))   # 1st = 10, median = 30, 99th = 40

---
## 3. Trim cutoffs
**Question.** Which hourly wages are too extreme to use? `hrwage_real` runs from about $0.0006 to $603,445 an hour, and those extremes do not look like real hourly pay. For each income year we find the 1st and 99th percentile of `hrwage_real`, weighted with `asecwt`, using **all workers** (full-time and part-time, men and women together). Wages below the 1st or above the 99th percentile are dropped. There is one set of cutoffs per year, and every table uses it.

In [ ]:
def cutoffs_for(data):                                     # one year of data
    return pd.Series({"p1": wquantile(data, "hrwage_real", 0.01), "p99": wquantile(data, "hrwage_real", 0.99)})   # low and high cutoff

cutoffs = df.groupby("year").apply(cutoffs_for, include_groups=False)   # one row per year, all workers
print(cutoffs.round(1))                                    # dollars per hour, 2025 dollars

In [ ]:
df["p1"] = df["year"].map(cutoffs["p1"])                   # each row gets its year's low cutoff
df["p99"] = df["year"].map(cutoffs["p99"])                 # and its year's high cutoff
df["keep"] = (df["hrwage_real"] >= df["p1"]) & (df["hrwage_real"] <= df["p99"])   # True = inside the cutoffs
df["trimmed"] = (~df["keep"]).astype(int)                  # 1 = dropped by the trim

kept = df[df["keep"]]                                      # workers inside the cutoffs
kept_ft = kept[kept["fulltime"] == 1]                      # full-time workers inside the cutoffs

Check: how much did the trim drop? For all workers, each end drops about 1%, so about 2% in total. For full-time workers the share can differ, because the cutoffs come from everyone.

In [ ]:
check = pd.DataFrame({
    "all workers": df.groupby("year").apply(wmean, "trimmed", include_groups=False) * 100,                          # % of people dropped
    "full-time": df[df["fulltime"] == 1].groupby("year").apply(wmean, "trimmed", include_groups=False) * 100,       # % of full-time workers dropped
})
print(check.round(1))

ft_total = (df["fulltime"] == 1).sum()                     # full-time workers in the file (unweighted rows)
ft_dropped = df.loc[df["fulltime"] == 1, "trimmed"].sum()  # of those, dropped by the trim
print("Rows dropped by the trim:", df["trimmed"].sum(), "of", len(df))
print("Full-time rows dropped by the trim:", ft_dropped, "of", ft_total)

---
## 4. Table 1: hourly wage by sex
**Question.** How much do women earn per hour compared with men, year by year?

**Who is in it?** Full-time workers inside the trim cutoffs. **What is in each cell?** Dollars per hour in 2025 dollars, weighted with `asecwt`. The gap is shown two ways: women's wage as a % of men's, and men minus women in dollars. Rows are income years.

In [ ]:
sex_order = ["Male", "Female"]                                                 # our column order
median_wage = kept_ft.groupby(["year", "female_label"]).apply(wquantile, "hrwage_real", 0.5, include_groups=False)   # weighted median
median_wage = median_wage.unstack()[sex_order]                                 # years down the rows, sex across
median_wage["Women as % of men"] = median_wage["Female"] / median_wage["Male"] * 100   # gap in percent
median_wage["Men minus women ($)"] = median_wage["Male"] - median_wage["Female"]       # gap in dollars
print("Table 1a. Median hourly wage of full-time workers, 2025 dollars")
print(median_wage.round(1))
print("Rows left out: no missing wage or sex;", ft_dropped, "of", ft_total, "full-time rows dropped by the trim")

In [ ]:
mean_wage = kept_ft.groupby(["year", "female_label"]).apply(wmean, "hrwage_real", include_groups=False)   # weighted mean
mean_wage = mean_wage.unstack()[sex_order]                                     # years down the rows, sex across
mean_wage["Women as % of men"] = mean_wage["Female"] / mean_wage["Male"] * 100         # gap in percent
mean_wage["Men minus women ($)"] = mean_wage["Male"] - mean_wage["Female"]             # gap in dollars
print("Table 1b. Mean hourly wage of full-time workers, 2025 dollars")
print(mean_wage.round(1))
print("Rows left out: no missing wage or sex;", ft_dropped, "of", ft_total, "full-time rows dropped by the trim")